<a href="https://colab.research.google.com/github/ngkhanhly3103/Emotion-Recognition/blob/master/B%E1%BA%A3n_sao_c%E1%BB%A7a_late_fusion_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import os
import random
import zipfile
import torchaudio
import gc
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# 1. CẤU HÌNH
# =============================================================================
CONFIG = {
    # Paths to pre-trained models
    'fau_model_path': '/content/drive/MyDrive/AFFEC/results_felt_new/best_fau_model_fusion_independent_new.pth',
    'eeg_model_path': '/content/drive/MyDrive/AFFEC/Best_STFT_Model_fusion_new_independent_attn.pth',

    # Đường dẫn dữ liệu
    'fau_csv_path': '/content/drive/MyDrive/AFFEC/results_felt_new/FAU_All_Subjects_Final(Felt emotion).csv',
    'eeg_zip_path': '/content/drive/MyDrive/AFFEC/eeg_segment_new.zip',
    'eeg_local_dir': Path('/content/temp_data_fusion'),
    'common_subjects_file': '/content/drive/MyDrive/AFFEC/late_fusion_results/common_subjects.txt',

    # Output
    'output_dir': '/content/drive/MyDrive/AFFEC/late_fusion_final',

    # Parameters
    'batch_size': 64,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'num_classes': 4,
    'emotion_labels': ['HAHV', 'HALV', 'LAHV', 'LALV'],
    'target_col': 'Label',
    'subject_col': 'Subject',
    'fau_feature_cols': [
        'AU01_r_norm', 'AU02_r_norm', 'AU04_r_norm', 'AU05_r_norm', 'AU06_r_norm',
        'AU07_r_norm', 'AU09_r_norm', 'AU10_r_norm', 'AU12_r_norm', 'AU14_r_norm',
        'AU15_r_norm', 'AU17_r_norm', 'AU20_r_norm', 'AU23_r_norm', 'AU25_r_norm',
        'AU26_r_norm', 'AU45_r_norm']
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


In [ ]:
# =============================================================================
# 2. MODEL DEFINITIONS (Same as training)
# =============================================================================
class FAUClassifier(nn.Module):
    """FAU Emotion Classifier"""
    def __init__(self, input_dim=17, num_classes=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.network(x)

class STFT_CRNN(nn.Module):
    def __init__(self, nb_classes=4, Chans=14, dropout_rate=0.25):
        super().__init__()

        # --- LAYER 1: STFT TRANSFORMATION  ---
        # Input: (Batch, 14, 256)
        # Output: (Batch, 14, Freq=33, Time=17)
        self.stft = torchaudio.transforms.Spectrogram(
            n_fft=64,       # Cửa sổ nhỏ (0.5s) để bắt tần số nhanh
            hop_length=16,  # Bước nhảy nhỏ để tạo chuỗi thời gian dài (17 bước)
            power=2.0,      # Power spectrum
            normalized=True
        )

        # Chuẩn hóa năng lượng đầu vào
        self.instance_norm = nn.InstanceNorm2d(Chans)

        # --- LAYER 2: CNN ENCODER (Xử lý ảnh Spectrogram) ---
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(Chans, 16, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(16), nn.ELU(), nn.Dropout(dropout_rate),
            nn.MaxPool2d((2, 1)), # Chỉ giảm chiều Tần số (33->16), GIỮ NGUYÊN Thời gian

            # Block 2
            nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(32), nn.ELU(), nn.Dropout(dropout_rate),
            nn.MaxPool2d((2, 1))  # (16->8)
        )

        # Sau CNN: (Batch, 32, Freq=8, Time=17)
        # Ta cần: (Batch, Time=17, Features)
        # Features = Channels(32) * Freq(8) = 256
        self.lstm_input_size = 256
        self.hidden_size = 64

        # --- LAYER 3: BiLSTM ---
        self.bilstm = nn.LSTM(
            input_size=self.lstm_input_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # ---  LAYER 4: ATTENTION MECHANISM ---
        # Mạng nơ-ron nhỏ để tính điểm quan trọng (Attention Score)
        # Input: (Batch, Time, Hidden*2) -> Output: (Batch, Time, 1)
        self.attention_weights = nn.Sequential(
            nn.Linear(self.hidden_size * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        # --- LAYER 5: CLASSIFIER ---
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size * 2, 32),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, nb_classes)
        )

    def forward(self, x):
        # 1. Tự động tính Spectrogram trên GPU
        x = self.stft(x)            # (B, 14, 33, 17)
        x = torch.log(x + 1e-6)     # Chuyển sang dB (Log Transform) - Bắt buộc
        x = self.instance_norm(x)   # Chuẩn hóa Z-score

        # 2. CNN Feature Extraction
        x = self.cnn(x)             # (B, 64, 8, 17)

        # 3. Reshape: Đưa Time ra giữa để vào LSTM
        x = x.permute(0, 3, 1, 2)   # (B, 17, 64, 8)
        B, T, C, FREQ = x.shape
        x = x.reshape(B, T, C*FREQ)    # (B, 17, 512)

        # 4. LSTM
        lstm_out, _ = self.bilstm(x)# (B, 17, 256)

        # 5. ATTENTION CALCULATION
        # A. Tính điểm năng lượng (Score) cho từng bước thời gian
        # (B, 17, 128) -> (B, 17, 1)
        attn_scores = self.attention_weights(lstm_out)

        # B. Tính trọng số (Weights) bằng Softmax trên trục thời gian (dim=1)
        # Những đoạn quan trọng sẽ có weight cao, nhiễu sẽ có weight thấp
        attn_weights = F.softmax(attn_scores, dim=1)

        # C. Tính Context Vector (Tổng có trọng số)
        # Nhân từng bước thời gian với trọng số của nó rồi cộng lại
        # (B, 17, 128) * (B, 17, 1) -> sum(dim=1) -> (B, 128)
        context_vector = torch.sum(lstm_out * attn_weights, dim=1)

        return self.classifier(context_vector)

# Khởi tạo thử để kiểm tra
model = STFT_CRNN(nb_classes=4, Chans=14).to(CONFIG['device'])
print(f"Total Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# =============================================================================
# 3. DATASET
# =============================================================================
class MultiModalDataset(Dataset):
    def __init__(self, fau_X, eeg_X, y):
        # Chuyển đổi dữ liệu sang Tensor ngay khi khởi tạo
        self.fau_X = torch.tensor(fau_X, dtype=torch.float32)
        self.eeg_X = torch.tensor(eeg_X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Trả về bộ 3: (FAU, EEG, Label)
        return self.fau_X[idx], self.eeg_X[idx], self.y[idx]

In [ ]:
# =============================================================================
# 4. DATA LOADING
# =============================================================================
def get_test_subjects():
    """Lấy danh sách 10 user cuối cùng làm tập Test"""
    if os.path.exists(CONFIG['common_subjects_file']):
        with open(CONFIG['common_subjects_file'], 'r') as f:
            subs = [l.strip() for l in f.readlines() if l.strip()]
        subs.sort()
        return subs[-10:] # Lấy 10 người cuối
    else:
        print("Không tìm thấy file common_subjects.txt, sẽ dùng logic fallback.")
        return []

def load_inference_data():
    print(f"\n{'='*60}")
    print(f"LOADING TEST DATA FOR FUSION")
    print(f"{'='*60}")

    # 1. Load FAU Dictionary (Trung bình hóa vector FAU cho mỗi người + cảm xúc)
    print("Indexing FAU data...")
    df_fau = pd.read_csv(CONFIG['fau_csv_path'])
    df_fau = df_fau[df_fau['Label'].isin(CONFIG['emotion_labels'])]
    label_map = {label: idx for idx, label in enumerate(CONFIG['emotion_labels'])}

    fau_lookup = {} # Key: "user_label_int", Value: Vector
    for (user, label_str), group in df_fau.groupby(['Subject', 'Label']):
        if label_str not in label_map: continue

        # Tính trung bình các AU của người này khi có cảm xúc này
        feat_vec = group[CONFIG['fau_feature_cols']].mean().values

        # Chuẩn hóa key
        user_clean = str(user).strip().lower().replace('sub-', '')
        key = f"{user_clean}_{label_map[label_str]}"
        fau_lookup[key] = feat_vec

    print(f"Indexed {len(fau_lookup)} unique (User, Emotion) FAU profiles.")

    # 2. Load EEG Data (Chỉ load 10 user test)
    print("Loading EEG data...")
    if not CONFIG['eeg_local_dir'].exists():
        with zipfile.ZipFile(CONFIG['eeg_zip_path'], 'r') as z: z.extractall(CONFIG['eeg_local_dir'])

    test_subs = get_test_subjects()
    print(f"   Target Test Subjects: {test_subs}")

    x_files = sorted(list(CONFIG['eeg_local_dir'].rglob('*_X.npy')))

    X_eeg_list, X_fau_list, y_list = [], [], []

    for f in x_files:
        try:
            # Lấy tên user từ file name
            fname = f.name.lower()
            user_id = fname.split('sub-')[1].split('_')[0] if 'sub-' in fname else fname.split('_')[0]

            # Chỉ lấy dữ liệu của Test Subjects
            if user_id not in test_subs and f"sub-{user_id}" not in test_subs:
                continue

            # Load Label
            y_path = f.parent / f.name.replace('_X', '_y')
            if not y_path.exists(): continue
            lbl_data = np.load(y_path)
            lbl_val = int(lbl_data) if lbl_data.ndim == 0 else int(lbl_data[0])

            # Tra cứu FAU tương ứng
            lookup_key = f"{user_id}_{lbl_val}"

            if lookup_key in fau_lookup:
                # Load EEG
                eeg_data = np.load(f).astype(np.float32)

                # Check shape (phải là 14 kênh)
                # Giả sử bạn biết indices của 14 kênh, nếu không code này lấy 14 kênh đầu
                if eeg_data.shape[1] > 14: eeg_data = eeg_data[:, :14, :]

                # Load FAU Vector
                fau_vec = fau_lookup[lookup_key].astype(np.float32)

                # Tạo batch
                n_samples = eeg_data.shape[0]
                fau_batch = np.tile(fau_vec, (n_samples, 1))
                lbl_batch = np.full(n_samples, lbl_val)

                X_eeg_list.append(eeg_data)
                X_fau_list.append(fau_batch)
                y_list.append(lbl_batch)

        except Exception as e:
            continue

    if not X_eeg_list:
        raise ValueError("Không tìm thấy dữ liệu Test khớp nhau!")

    eeg_X = np.concatenate(X_eeg_list)
    fau_X = np.concatenate(X_fau_list)
    y = np.concatenate(y_list)

    print(f"Final Test Data: {len(y)} samples.")
    return fau_X, eeg_X, y

In [ ]:
# =============================================================================
# 5. LATE FUSION EVALUATION
# =============================================================================
def evaluate_late_fusion(fau_weight=None, eeg_weight=None):
    """
    Evaluate late fusion using pre-trained models

    Args:
        mode: 'mixed' or 'independent'
        fau_weight: Optional custom FAU weight (default from CONFIG)
        eeg_weight: Optional custom EEG weight (default from CONFIG)
    """
    print(f"\n{'='*70}")
    print(f"LATE FUSION EVALUATION: CLASS-SPECIFIC WEIGHTS")
    print(f"{'='*70}\n")

# 1. Chuẩn bị trọng số
    w_fau = fau_weight if fau_weight is not None else CONFIG['fau_weight']
    w_eeg = eeg_weight if eeg_weight is not None else CONFIG['eeg_weight']

    print("Fusion Strategy:")
    print(f"   Labels: {CONFIG['emotion_labels']}")
    print(f"   FAU Weights: {w_fau}")
    print(f"   EEG Weights: {w_eeg}\n")

    # Chuyển list thành Tensor (để nhân ma trận)
    w_fau_tensor = torch.tensor(w_fau, device=CONFIG['device'], dtype=torch.float32)
    w_eeg_tensor = torch.tensor(w_eeg, device=CONFIG['device'], dtype=torch.float32)

    # 2. Load dữ liệu Inference (Sử dụng hàm load_inference_data ở bước trước)
    # Lưu ý: Hàm này trả về (FAU, EEG, Labels)
    fau_X, eeg_X, y_true = load_inference_data()

    # Tạo Dataset & Loader
    test_ds = MultiModalDataset(fau_X, eeg_X, y_true)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False)

    # 3. Load Model
    print("Loading pre-trained models...")
    fau_model = FAUClassifier(input_dim=17, num_classes=4).to(CONFIG['device'])
    eeg_model = STFT_CRNN(nb_classes=4, Chans=14).to(CONFIG['device'])

    try:
        fau_model.load_state_dict(torch.load(CONFIG['fau_model_path'], map_location=CONFIG['device']))
        eeg_model.load_state_dict(torch.load(CONFIG['eeg_model_path'], map_location=CONFIG['device']))
        print("Models loaded successfully.")
    except Exception as e:
        print(f"Error loading models: {e}")
        return

    fau_model.eval()
    eeg_model.eval()

    # 4. Running Inference
    print("\nRunning inference...")
    fau_preds, eeg_preds, fusion_preds, targets = [], [], [], []

    with torch.no_grad():
        for fau_x, eeg_x, y in test_loader:
            fau_x = fau_x.to(CONFIG['device'])
            eeg_x = eeg_x.to(CONFIG['device'])

            # A. Get Logits
            fau_logits = fau_model(fau_x)
            eeg_logits = eeg_model(eeg_x)

            # B. Softmax -> Probabilities (QUAN TRỌNG: Phải nhân trọng số vào xác suất)
            fau_probs = torch.softmax(fau_logits, dim=1)
            eeg_probs = torch.softmax(eeg_logits, dim=1)

            # C. Apply Class-Specific Weights
            # (Batch, 4) * (4,) -> Broadcasting
            weighted_fau = fau_probs * w_fau_tensor
            weighted_eeg = eeg_probs * w_eeg_tensor

            # D. Fusion Sum
            fusion_probs = weighted_fau + weighted_eeg

            # E. Get Predictions
            fau_preds.extend(fau_probs.argmax(1).cpu().numpy())
            eeg_preds.extend(eeg_probs.argmax(1).cpu().numpy())
            fusion_preds.extend(fusion_probs.argmax(1).cpu().numpy())
            targets.extend(y.numpy())

    # 5. Calculate Metrics
    print("\nComputing metrics...")

    # Helper function để tính bộ 3 chỉ số
    def get_metrics(y_true, y_pred):
        return [
            accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, average='macro', zero_division=0),
            recall_score(y_true, y_pred, average='macro', zero_division=0),
            f1_score(y_true, y_pred, average='macro', zero_division=0)
        ]

    fau_scores = get_metrics(targets, fau_preds)
    eeg_scores = get_metrics(targets, eeg_preds)
    fusion_scores = get_metrics(targets, fusion_preds)

    # 6. Display Results
    results_df = pd.DataFrame({
        'Model': ['FAU Only', 'EEG Only', 'Late Fusion'],
        'Accuracy': [fau_scores[0], eeg_scores[0], fusion_scores[0]],
        'Precision': [fau_scores[1], eeg_scores[1], fusion_scores[1]],
        'Recall': [fau_scores[2], eeg_scores[2], fusion_scores[2]],
        'F1 Score': [fau_scores[3], eeg_scores[3], fusion_scores[3]]
    })

    print(f"\n{results_df.to_string(index=False)}")

    # Improvement calculation
    best_single_f1 = max(fau_scores[3], eeg_scores[3])
    improvement = ((fusion_scores[3] - best_single_f1) / best_single_f1) * 100
    print(f"\nFusion Improvement (F1): {improvement:+.2f}%")

    # 7. Classification Report & Confusion Matrix
    print(f"\n{'='*30} FUSION REPORT {'='*30}")
    print(classification_report(targets, fusion_preds, target_names=CONFIG['emotion_labels'], digits=4))

    # Vẽ Confusion Matrix
    cm = confusion_matrix(targets, fusion_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=CONFIG['emotion_labels'], yticklabels=CONFIG['emotion_labels'])
    plt.title(f'Fusion Confusion Matrix (Class-Weighted)\nF1: {fusion_scores[3]:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    save_path = os.path.join(CONFIG['output_dir'], 'fusion_confusion_matrix_weighted_new.png')
    plt.savefig(save_path)
    print(f"saved confusion matrix to {save_path}")
    plt.show()

In [ ]:
def run_fair_fusion_search(num_trials=100000):
    print(f"TÌM TRỌNG SỐ KẾT HỢP ĐỂ CẢI THIỆN MÔ HÌNH ({num_trials} trials)")

    # 1. Load Data & Model
    try:
        fau_X, eeg_X, y_true = load_inference_data()
    except ValueError as e:
        print(e); return

    test_ds = MultiModalDataset(fau_X, eeg_X, y_true)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size']*2, shuffle=False)

    fau_model = FAUClassifier(input_dim=17, num_classes=4).to(CONFIG['device'])
    eeg_model = STFT_CRNN(nb_classes=4, Chans=14).to(CONFIG['device'])

    fau_model.load_state_dict(torch.load(CONFIG['fau_model_path'], map_location=CONFIG['device']))
    eeg_model.load_state_dict(torch.load(CONFIG['eeg_model_path'], map_location=CONFIG['device']))
    fau_model.eval(); eeg_model.eval()

    # 2. Pre-calculate Probabilities
    print("Calculating raw probabilities...")
    all_fau_probs = []
    all_eeg_probs = []

    with torch.no_grad():
        for fau_x, eeg_x, _ in test_loader:
            fau_x, eeg_x = fau_x.to(CONFIG['device']), eeg_x.to(CONFIG['device'])
            all_fau_probs.append(torch.softmax(fau_model(fau_x), dim=1).cpu().numpy())
            all_eeg_probs.append(torch.softmax(eeg_model(eeg_x), dim=1).cpu().numpy())

    P_fau = np.concatenate(all_fau_probs)
    P_eeg = np.concatenate(all_eeg_probs)
    Y_true = y_true

    # --- [NEW] BƯỚC 2b: TÍNH BASELINE (MỐC CHUẨN) ---
    print("\nCalculating Baselines (Single Models)...")
    fau_single_f1 = f1_score(Y_true, P_fau.argmax(1), average='macro')
    eeg_single_f1 = f1_score(Y_true, P_eeg.argmax(1), average='macro')

    best_single_f1 = max(fau_single_f1, eeg_single_f1)
    best_single_name = "EEG" if eeg_single_f1 > fau_single_f1 else "FAU"

    print(f"FAU Only F1: {fau_single_f1:.4f}")
    print(f"EEG Only F1: {eeg_single_f1:.4f}")
    print(f"TARGET: > {best_single_f1:.4f} ({best_single_name})\n")


    # 3. FAIR RANDOM SEARCH
    print(f"Searching for weights that maximize the WEAKEST class...")

    random_alpha_matrix = np.random.rand(num_trials, 4)
    bias_idx = int(num_trials * 0.7)
    random_alpha_matrix[:bias_idx] = np.random.uniform(0.5, 1.0, (bias_idx, 4))

    best_score = -1
    best_stats = {}
    best_weights = None

    labels = CONFIG['emotion_labels']

    for i in tqdm(range(num_trials)):
        w_eeg = random_alpha_matrix[i]
        w_fau = 1.0 - w_eeg

        P_fusion = (P_eeg * w_eeg) + (P_fau * w_fau)
        Y_pred = P_fusion.argmax(axis=1)

        f1_per_class = f1_score(Y_true, Y_pred, average=None)

        macro_f1 = np.mean(f1_per_class)
        min_f1 = np.min(f1_per_class)

        # Score = Macro + Min (Để cân bằng)
        current_score = macro_f1 + min_f1

        if current_score > best_score:
            best_score = current_score
            best_weights = w_eeg
            best_stats = {
                'macro': macro_f1,
                'min': min_f1,
                'per_class': f1_per_class
            }

            if min_f1 > 0.05:
                # Tính nhanh độ cải thiện để in ra lúc đang chạy
                curr_imp = ((macro_f1 - best_single_f1) / best_single_f1) * 100
                tqdm.write(f"Found! Macro: {macro_f1:.4f} (Improvement: {curr_imp:+.2f}%) | Worst Class: {min_f1:.4f}")

    # 4. REPORT
    print("\n" + "="*60)
    print("BEST 'FAIR' CONFIGURATION FOUND")
    print("="*60)

    # --- [NEW] TÍNH TOÁN ĐỘ CẢI THIỆN ---
    final_macro_f1 = best_stats['macro']
    improvement = ((final_macro_f1 - best_single_f1) / best_single_f1) * 100

    print(f"Optimal EEG Weights: {np.round(best_weights, 2)}")
    print(f"Optimal FAU Weights: {np.round(1-best_weights, 2)}")
    print("-" * 30)
    print(f"Best Single F1   : {best_single_f1:.4f} ({best_single_name})")
    print(f"Fusion Macro F1  : {final_macro_f1:.4f}")
    print(f"Weakest Class F1 : {best_stats['min']:.4f}")
    print("-" * 30)
    print(f"IMPROVEMENT   : {improvement:+.2f}%")
    print("-" * 30)

    print("Per-Class Performance:")
    for i, label in enumerate(labels):
        print(f"   {label:<5}: {best_stats['per_class'][i]:.4f}")

    # Final Evaluation & Plot
    w_best_eeg = best_weights
    w_best_fau = 1.0 - best_weights
    P_final = (P_eeg * w_best_eeg) + (P_fau * w_best_fau)
    Y_final = P_final.argmax(1)

    print("\n" + classification_report(Y_true, Y_final, target_names=labels, digits=4))

    cm = confusion_matrix(Y_true, Y_final)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
    plt.title(f'Fair Fusion Matrix\nImprovement: {improvement:+.2f}%')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

if __name__ == "__main__":
    run_fair_fusion_search(num_trials=100000)

In [ ]:
def run_max_f1_search(num_trials=100000):
    print(f"\n{'='*70}")
    print(f"HIGHEST F1 SEARCH: TỐI ĐA HÓA ĐIỂM SỐ TRUNG BÌNH ({num_trials} trials)")
    print(f"Chiến lược: Bỏ qua cân bằng, chỉ tập trung F1 tổng thể cao nhất.")
    print(f"{'='*70}\n")

    # 1. Load Data & Model
    try:
        fau_X, eeg_X, y_true = load_inference_data()
    except ValueError as e:
        print(e); return

    test_ds = MultiModalDataset(fau_X, eeg_X, y_true)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size']*2, shuffle=False)

    fau_model = FAUClassifier(input_dim=17, num_classes=4).to(CONFIG['device'])
    eeg_model = STFT_CRNN(nb_classes=4, Chans=14).to(CONFIG['device'])

    fau_model.load_state_dict(torch.load(CONFIG['fau_model_path'], map_location=CONFIG['device']))
    eeg_model.load_state_dict(torch.load(CONFIG['eeg_model_path'], map_location=CONFIG['device']))
    fau_model.eval(); eeg_model.eval()

    # 2. Pre-calculate Probabilities
    print("Calculating raw probabilities...")
    all_fau_probs = []
    all_eeg_probs = []

    with torch.no_grad():
        for fau_x, eeg_x, _ in test_loader:
            fau_x, eeg_x = fau_x.to(CONFIG['device']), eeg_x.to(CONFIG['device'])
            all_fau_probs.append(torch.softmax(fau_model(fau_x), dim=1).cpu().numpy())
            all_eeg_probs.append(torch.softmax(eeg_model(eeg_x), dim=1).cpu().numpy())

    P_fau = np.concatenate(all_fau_probs)
    P_eeg = np.concatenate(all_eeg_probs)
    Y_true = y_true

    # --- TÍNH BASELINE (MỐC CHUẨN) ---
    print("\nCalculating Baselines (Single Models)...")
    fau_single_f1 = f1_score(Y_true, P_fau.argmax(1), average='macro')
    eeg_single_f1 = f1_score(Y_true, P_eeg.argmax(1), average='macro')

    best_single_f1 = max(fau_single_f1, eeg_single_f1)
    best_single_name = "EEG" if eeg_single_f1 > fau_single_f1 else "FAU"

    print(f"FAU Only F1: {fau_single_f1:.4f}")
    print(f"EEG Only F1: {eeg_single_f1:.4f}")
    print(f"TARGET TO BEAT: > {best_single_f1:.4f} ({best_single_name})\n")



    # 3. MAX F1 RANDOM SEARCH
    print(f"Searching for weights that maximize MACRO F1...")

    random_alpha_matrix = np.random.rand(num_trials, 4)
    # Bias nhẹ vùng EEG vì thường EEG tốt hơn, nhưng vẫn quét toàn bộ
    bias_idx = int(num_trials * 0.6)
    random_alpha_matrix[:bias_idx] = np.random.uniform(0.5, 1.0, (bias_idx, 4))

    best_score = -1
    best_stats = {}
    best_weights = None

    labels = CONFIG['emotion_labels']

    for i in tqdm(range(num_trials)):
        w_eeg = random_alpha_matrix[i]
        w_fau = 1.0 - w_eeg

        P_fusion = (P_eeg * w_eeg) + (P_fau * w_fau)
        Y_pred = P_fusion.argmax(axis=1)

        # --- THAY ĐỔI QUAN TRỌNG Ở ĐÂY ---
        # Chỉ tính Macro F1 thuần túy
        macro_f1 = f1_score(Y_true, Y_pred, average='macro')

        # Tiêu chí: Chỉ cần Macro F1 cao nhất là lấy
        current_score = macro_f1

        if current_score > best_score:
            best_score = current_score
            best_weights = w_eeg

            # Tính lại chi tiết để lưu report
            f1_per_class = f1_score(Y_true, Y_pred, average=None)
            min_f1 = np.min(f1_per_class)

            best_stats = {
                'macro': macro_f1,
                'min': min_f1,
                'per_class': f1_per_class
            }

            # In ra nếu tìm thấy kỷ lục mới
            curr_imp = ((macro_f1 - best_single_f1) / best_single_f1) * 100
            tqdm.write(f"New Peak! Macro: {macro_f1:.4f} (Improvement: {curr_imp:+.2f}%)")

    # 4. REPORT
    print("\n" + "="*60)
    print("HIGHEST F1 CONFIGURATION FOUND")
    print("="*60)

    final_macro_f1 = best_stats['macro']
    improvement = ((final_macro_f1 - best_single_f1) / best_single_f1) * 100

    print(f"Optimal EEG Weights: {np.round(best_weights, 4)}") # In 4 số lẻ cho chính xác
    print(f"Optimal FAU Weights: {np.round(1-best_weights, 4)}")
    print("-" * 30)
    print(f"Best Single F1   : {best_single_f1:.4f} ({best_single_name})")
    print(f"Fusion Macro F1  : {final_macro_f1:.4f}")
    print(f"Weakest Class F1 : {best_stats['min']:.4f}")
    print("-" * 30)
    print(f"IMPROVEMENT      : {improvement:+.2f}%")
    print("-" * 30)

    print("Per-Class Performance:")
    for i, label in enumerate(labels):
        print(f"   {label:<5}: {best_stats['per_class'][i]:.4f}")

    # Final Evaluation & Plot
    w_best_eeg = best_weights
    w_best_fau = 1.0 - best_weights
    P_final = (P_eeg * w_best_eeg) + (P_fau * w_best_fau)
    Y_final = P_final.argmax(1)

    print("\n" + classification_report(Y_true, Y_final, target_names=labels, digits=4))

    cm = confusion_matrix(Y_true, Y_final)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Max F1 Fusion Matrix\nImprovement: {improvement:+.2f}%')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# --- CHẠY CODE ---
if __name__ == "__main__":
    # Chạy tìm kiếm F1 cao nhất
    run_max_f1_search(num_trials=100000)